<a href="https://colab.research.google.com/github/Zahra-Aliyeva/ai-learning-projects/blob/main/customer_feedback_analyzer_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

In [4]:
from getpass import getpass
from google import genai

api_key = getpass("API açarını yapışdırın və Enter basın: ")
client = genai.Client(api_key=api_key)

cavab = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Salam! Özünü bir cümlə ilə təqdim et.",
)
print(cavab.text)

API açarını yapışdırın və Enter basın: ··········
Salam, mən suallarınızı cavablandırmaq və müxtəlif tapşırıqlarda sizə kömək etmək üçün yaradılmış süni intellekt asistentiyəm.


In [6]:
import time
from google.genai import types, errors

def sual_ver(sual, qayda=None):
    for cehd in range(5):
        try:
            config = types.GenerateContentConfig(system_instruction=qayda) if qayda else None
            cavab = client.models.generate_content(
                model="gemini-3.6-flash", contents=sual, config=config
            )
            return cavab.text
        except errors.ServerError:
            print(f"Cəhd {cehd + 1}: server məşğuldur, 10 saniyə gözləyirəm...")
            time.sleep(10)
    return "Server hələ də məşğuldur. Bir az sonra yenidən sınayın."

sual = "LLM nədir?"

print("QAYDASIZ:\n", sual_ver(sual))
print("\nQAYDALI:\n", sual_ver(
    sual,
    "Sən müəllimsən. Azərbaycan dilində, 10 yaşlı uşağa izah edirmiş kimi, maksimum 3 cümlə ilə cavab ver."
))

QAYDASIZ:
 **LLM** — **Large Language Model** sözlərinin qısaldılmasıdır və azərbaycanca **"Böyük Dil Modeli"** deməkdir.

Bu, insan dilini anlamaq, təhlil etmək və təbii şəkildə yeni mətnlər yaratmaq üçün böyük həcmdə məlumatlar üzərində təlim keçmiş **Süni İntellekt (AI)** texnologiyasıdır.

LLM-lər haqqında bilməli olduğunuz əsas məqamlar bunlardır:

### 1. Necə işləyir?
LLM-lər dərin öyrənmə (deep learning) və neyron şəbəkələri (xüsusilə *Transformer* arxitekturası) əsasında çalışır. Onlara internetdəki milyardlarla mətn (kitablar, məqalələr, veb-saytlar) daxil edilir. Modeli belə təsəvvür edə bilərsiniz: o, özündən əvvəl gələn sözlərə baxaraq **"növbəti ən məntiqli sözün nə olacağını"** riyazi ehtimallarla təxmin edir.

### 2. LLM-lər nə edə bilir?
* **Mətn yaratmaq:** Esse, e-poçt, hekayə, şeir və ya hesabat yazmaq.
* **Mətnləri xülasə etmək:** Uzun bir kitabı və ya məqaləni bir neçə cümlə ilə xülasə etmək.
* **Tərcümə:** Dillər arasında yüksək dəqiqliklə tərcümə etmək.
* **Suall

In [9]:
import time
from getpass import getpass
from google import genai
from google.genai import types, errors

api_key = getpass("API açarını yapışdırın və Enter basın: ").strip()
client = genai.Client(api_key=api_key)
MODEL = "gemini-3.6-flash"


def sual_ver(sual, qayda):
    """Modelə sual göndərir. Server məşğul olsa və ya limit dolsa, gözləyib təkrar cəhd edir."""
    for cehd in range(5):
        try:
            cavab = client.models.generate_content(
                model=MODEL,
                contents=sual,
                config=types.GenerateContentConfig(system_instruction=qayda, temperature=0),
            )
            return cavab.text.strip()
        except errors.ServerError:
            print("Server məşğuldur, 10 saniyə gözləyirəm...")
            time.sleep(10)
        except errors.ClientError as xeta:
            if xeta.code == 429:
                print("Pulsuz limit doldu, 60 saniyə gözləyirəm...")
                time.sleep(60)
            else:
                raise
    return "XƏTA"


# Xəyali müştəri rəyləri (özünüzünkülərlə əvəz edə bilərsiniz)
reyler = [
    "Mobil tətbiq çox rahatdır, köçürmə bir dəqiqəyə oldu.",
    "Filialda 40 dəqiqə növbə gözlədim, çox narazıyam.",
    "Başqa banka köçürmə komissiyası çox yüksəkdir.",
    "Operator çox nəzakətli idi, problemimi tez həll etdi.",
    "Tətbiq tez-tez donur və giriş kodu gəlmir.",
    "Kart 3 gün ərzində gəldi, əla xidmətdir.",
    "Çağrı mərkəzi cavab vermir, 20 dəqiqə gözlədim.",
    "Normal bankdır, xüsusi bir şey yoxdur.",
]

netice = {"müsbət": 0, "neytral": 0, "mənfi": 0}
menfi_reyler = []

# 1) Bütün rəyləri BİR sorğuda AI-a oxutmaq
qayda1 = (
    "Hər müştəri rəyinin əhval-ruhiyyəsini təyin et. "
    "Cavabı hər rəy üçün ayrı sətirdə, yalnız bu formatda yaz: nömrə: söz "
    "(söz yalnız müsbət, neytral və ya mənfi ola bilər). Başqa heç nə yazma."
)
nomreli = "\n".join(f"{i + 1}: {r}" for i, r in enumerate(reyler))
cavab = sual_ver(nomreli, qayda1)

# 2) AI-nın cavabını oxuyub saymaq
for setir in cavab.splitlines():
    if ":" not in setir:
        continue
    nomre, hiss = setir.split(":", 1)
    nomre = nomre.strip(" *")
    if not nomre.isdigit() or not (1 <= int(nomre) <= len(reyler)):
        continue
    hiss = hiss.lower().strip(" .!*")
    if hiss not in netice:
        hiss = "neytral"
    rey = reyler[int(nomre) - 1]
    netice[hiss] += 1
    if hiss == "mənfi":
        menfi_reyler.append(rey)
    print(f"{hiss:8} | {rey}")

# 3) Faiz hesablamaq
print("\n=== NƏTİCƏ ===")
for ad, say in netice.items():
    print(f"{ad}: {say} rəy ({round(say / len(reyler) * 100)}%)")

# 4) Mənfi rəylərə əsasən AI hesabatı
if menfi_reyler:
    qayda2 = "Sən biznes analitiksən. Yalnız verilən rəylərə əsaslan, uydurma. Azərbaycan dilində, qısa yaz."
    metn = "Mənfi rəylər:\n" + "\n".join(menfi_reyler) + "\n\nƏsas problemləri və 2 konkret tövsiyəni yaz."
    print("\n=== AI HESABATI ===")
    print(sual_ver(metn, qayda2))

API açarını yapışdırın və Enter basın: ··········
müsbət   | Mobil tətbiq çox rahatdır, köçürmə bir dəqiqəyə oldu.
mənfi    | Filialda 40 dəqiqə növbə gözlədim, çox narazıyam.
mənfi    | Başqa banka köçürmə komissiyası çox yüksəkdir.
müsbət   | Operator çox nəzakətli idi, problemimi tez həll etdi.
mənfi    | Tətbiq tez-tez donur və giriş kodu gəlmir.
müsbət   | Kart 3 gün ərzində gəldi, əla xidmətdir.
mənfi    | Çağrı mərkəzi cavab vermir, 20 dəqiqə gözlədim.
neytral  | Normal bankdır, xüsusi bir şey yoxdur.

=== NƏTİCƏ ===
müsbət: 3 rəy (38%)
neytral: 1 rəy (12%)
mənfi: 4 rəy (50%)

=== AI HESABATI ===
Müştəri rəyləri əsasında aparılan təhlil:

**Əsas problemlər:**
1. **Uzun gözləmə müddəti:** Filialda (40 dəqiqə) və çağrı mərkəzində (20 dəqiqə) xidmətin həddindən artıq ləngiməsi.
2. **Mobil tətbiqin texniki xətaları:** Tətbiqin donması və giriş üçün doğrulama kodunun gəlməməsi.
3. **Maliyyə narazılığı:** Başqa banklara köçürmə komissiyasının yüksək olması.

---

**2 Konkret Tövsiyə:*